In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from typing import List

def logbook_to_dataframe(log) -> pd.DataFrame:
    """Convierte el logbook de DEAP a DataFrame con columnas gen, min, avg, max, std (si existen)."""
    cols = ["gen"]
    data = {"gen": log.select("gen")}
    for stat in ["min", "avg", "max", "std"]:
        try:
            data[stat] = log.select(stat)
            cols.append(stat)
        except Exception:
            pass
    df = pd.DataFrame(data, columns=cols)
    return df

def plot_evolution_threshold(df_stats: pd.DataFrame,
                             filename: str = "evolucion_fitness.png",
                             threshold: float = 1000.0,
                             replacement_value: float = 300.0):
    """
    Grafica min/avg/max por generación. Cualquier valor > threshold
    se reemplaza por replacement_value únicamente para la visualización.
    """
    if df_stats.empty:
        print("No hay estadísticas para graficar.")
        return

    gens  = df_stats["gen"].to_numpy()
    cols = [c for c in ["min", "avg", "max"] if c in df_stats.columns]

    plt.figure()
    for c in cols:
        arr = df_stats[c].to_numpy(dtype=float).copy()
        mask = np.isfinite(arr) & (arr > threshold)
        arr[mask] = replacement_value
        plt.plot(gens, arr, label=c)

    plt.xlabel("Generación")
    plt.ylabel("Fitness")
    plt.title(f"Evolución del fitness (>{threshold:g} → {replacement_value:g})")
    plt.legend()
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.close()
    print(f"Gráfica de evolución guardada en: {filename}")


def solution_dataframe(best_indices: List[int], hospitals_csv: str = "Hospitales_Con_Capacidad.csv",
                       name_col: str = "nombre", locality_col: str = "localidad") -> pd.DataFrame:
    """
    Devuelve un DataFrame con columnas [nombre, localidad] de los hospitales seleccionados (por índice).
    Asume que el orden de las filas del CSV coincide con el índice de los genes.
    """
    dfh = pd.read_csv(hospitals_csv)
    needed_cols = [name_col, locality_col]
    for col in needed_cols:
        if col not in dfh.columns:
            raise ValueError(f"En '{hospitals_csv}' no se encuentra la columna '{col}'.")
    out = dfh.loc[best_indices, needed_cols].reset_index(drop=True)
    return out



In [ ]:
# ============================
#  p-Centro capacitado (V-R)
#  GA con DEAP – Plantilla
# ============================

# Requisitos:
# pip install deap pandas numpy

import random
import math
from typing import List, Tuple, Dict

import numpy as np
import pandas as pd

from deap import base, creator, tools, algorithms

# -----------------------------------------------------------
# 1) CARGA Y PREPROCESADO DE DATOS
# -----------------------------------------------------------

def load_data(
    dist_csv: str = "time_ciudad_hospital_min.csv",
    cities_csv: str = "Ciudades_Con_Demanda.csv",
    hospitals_csv: str = "Hospitales_Con_Capacidad.csv",
    city_id_col: str = None,         # si tienes IDs explícitos, indícalos; si no, se usa el orden del CSV
    hosp_id_col: str = None
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, List, List]:
    """
    Carga:
      - D: matriz |V| x |H| con distancias (min).
      - q: vector |V| con demandas.
      - C: vector |H| con capacidades.
    Devuelve además listas con IDs (si existen) para trazabilidad.
    """
    # Matriz de distancias: filas=ciudades, columnas=hospitales
    D = pd.read_csv(dist_csv, header=None).values.astype(float)  # ajusta header=None según tus archivos

    df_cities = pd.read_csv(cities_csv)
    df_hosp = pd.read_csv(hospitals_csv)

    if 'q' not in df_cities.columns:
        raise ValueError("El CSV de municipios debe tener columna 'q' con las demandas.")
    if 'C' not in df_hosp.columns:
        raise ValueError("El CSV de hospitales debe tener columna 'C' con las capacidades.")

    q = df_cities['q'].values.astype(float)
    C = df_hosp['C'].values.astype(float)

    # IDs opcionales (para auditoría)
    city_ids = df_cities[city_id_col].tolist() if city_id_col and city_id_col in df_cities.columns else list(range(len(q)))
    hosp_ids = df_hosp[hosp_id_col].tolist() if hosp_id_col and hosp_id_col in df_hosp.columns else list(range(len(C)))

    # Chequeos básicos
    nV, nH = D.shape
    if nV != len(q):
        raise ValueError(f"Las filas de la matriz de distancias ({nV}) deben coincidir con |V|=len(q) ({len(q)}).")
    if nH != len(C):
        raise ValueError(f"Las columnas de la matriz de distancias ({nH}) deben coincidir con |H|=len(C) ({len(C)}).")

    return D, q, C, city_ids, hosp_ids


# -----------------------------------------------------------
# 2) REPRESENTACIÓN Y OPERADORES DE INDIVIDUO
# -----------------------------------------------------------

def make_random_individual(nH: int, p: int) -> List[int]:
    """ Devuelve una lista ordenada de índices de hospitales abiertos de tamaño p (sin repetición). """
    ind = random.sample(range(nH), p)
    ind.sort()
    return ind

def cx_set_based(ind1, ind2, nH: int, p: int):
    """
    Cruce específico para subconjuntos (evita duplicados y mantiene tamaño p).
    Devuelve hijos del MISMO TIPO que los padres (creator.Individual).
    """
    parent1 = list(ind1)
    parent2 = list(ind2)

    set1, set2 = set(parent1), set(parent2)
    common = list(set1.intersection(set2))
    only1 = list(set1 - set2)
    only2 = list(set2 - set1)

    # Hijo 1
    child1 = common.copy()
    random.shuffle(only1)
    random.shuffle(only2)
    pool1 = common + only1 + only2
    for g in pool1:
        if g not in child1:
            child1.append(g)
        if len(child1) == p:
            break

    # Hijo 2
    child2 = common.copy()
    pool2 = common + only2 + only1
    for g in pool2:
        if g not in child2:
            child2.append(g)
        if len(child2) == p:
            break

    child1.sort()
    child2.sort()

    # *** DEVOLVER MISMO TIPO ***
    cls = type(ind1)  # normalmente creator.Individual
    return cls(child1), cls(child2)


def mut_replace_gene(individual: List[int], nH: int, p: int, indpb: float = 0.2) -> Tuple[List[int]]:
    """
    Mutación: con prob indpb, sustituye un hospital por otro no incluido.
    Garantiza tamaño p y unicidad.
    """
    current = set(individual)
    all_idx = set(range(nH))
    free = list(all_idx - current)

    for pos in range(p):
        if random.random() < indpb and free:
            new_gene = random.choice(free)
            free.remove(new_gene)
            free.append(individual[pos])
            individual[pos] = new_gene

    individual.sort()
    return (individual,)

# -----------------------------------------------------------
# 3) DECODIFICADOR / ASIGNACIÓN (respeta capacidades si puede)
# -----------------------------------------------------------

def greedy_assignment_with_capacities(
    D: np.ndarray, q: np.ndarray, C: np.ndarray, open_idx: List[int]
) -> Tuple[np.ndarray, float, Dict[int, float], float, int]:
    """
    Asigna cada ciudad al hospital abierto más cercano respetando capacidades mediante un greedy:
      - Inicializa capacidad restante.
      - Ordena ciudades por distancia mínima al conjunto abierto (o por demanda, si prefieres).
      - Para cada ciudad, intenta asignar al hospital más cercano con capacidad suficiente.
      - Si no cabe, intenta el segundo más cercano, etc.
    Devuelve:
      assign[j] = lista de ciudades asignadas (implícito vía 'assign_of_city')
      Z = máximo d(i, j) de asignaciones realizadas
      cap_left[j] = capacidad remanente
      penalty_unserved = suma de demandas sin asignar (si quedó alguna)
      num_unserved = número de ciudades sin asignación
    """
    V, H = D.shape
    open_set = set(open_idx)

    # Capacidad restante
    cap_left = {j: float(C[j]) for j in open_set}

    # Precompute: para cada i, lista de hospitales abiertos ordenados por distancia
    nearest_open = {}
    for i in range(V):
        pairs = [(j, D[i, j]) for j in open_set]
        pairs.sort(key=lambda x: x[1])
        nearest_open[i] = pairs

    assign_of_city = -np.ones(V, dtype=int)
    Z = 0.0
    penalty_unserved = 0.0
    num_unserved = 0

    # Estrategias de ordenación de ciudades:
    #  a) Por demanda descendente (grandes primero) -> mejor usa capacidad
    #  b) Por distancia al más cercano (peores casos primero)
    # Aquí: demanda descendente
    city_order = list(range(V))
    city_order.sort(key=lambda i: q[i], reverse=True)

    for i in city_order:
        assigned = False
        for j, dij in nearest_open[i]:
            needed = q[i]
            if cap_left[j] >= needed:
                cap_left[j] -= needed
                assign_of_city[i] = j
                if dij > Z:
                    Z = dij
                assigned = True
                break

        if not assigned:
            # no hubo capacidad en ninguno -> penaliza
            penalty_unserved += q[i]
            num_unserved += 1

    return assign_of_city, Z, cap_left, penalty_unserved, num_unserved

# -----------------------------------------------------------
# 4) FUNCIÓN DE FITNESS (minimiza Z + penalizaciones)
# -----------------------------------------------------------

def fitness_function_factory(D: np.ndarray, q: np.ndarray, C: np.ndarray, bigM: float = 1e6, unserved_penalty: float = 1.0):
    """
    Crea una función fitness que:
      - Calcula asignación greedy con capacidades.
      - Devuelve (Z + penalizaciones,),
      donde:
        * Z = máximo de distancias de las ciudades servidas.
        * penalización = bigM si no se abre exactamente p (no debería ocurrir con nuestra codificación),
                        + unserved_penalty * (demanda no servida).
    Ajusta unserved_penalty para endurecer o suavizar soluciones parcialmente inviables.
    """
    V, H = D.shape

    def fitness(individual: List[int]) -> Tuple[float]:
        # (opcional) sanity check
        p = len(individual)
        if len(set(individual)) != p:
            return (bigM,)  # duplicados -> inválido

        assign, Z, cap_left, penalty_unserved, num_unserved = greedy_assignment_with_capacities(D, q, C, individual)
        # Penalización por demanda no atendida (puedes multiplicar por un factor grande)
        penalty = unserved_penalty * penalty_unserved
        return (Z + penalty,)

    return fitness

# -----------------------------------------------------------
# 5) CONFIGURACIÓN DE DEAP (Toolbox)
# -----------------------------------------------------------

def build_deap_toolbox(D: np.ndarray, q: np.ndarray, C: np.ndarray, p: int, seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)

    V, H = D.shape

    # Crear tipos
    if "FitnessMin" not in creator.__dict__:
        creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
    if "Individual" not in creator.__dict__:
        creator.create("Individual", list, fitness=creator.FitnessMin)

    toolbox = base.Toolbox()
    #toolbox.register("clone", tools.clone)

    # Registro de generador de individuos y población
    toolbox.register("individual", tools.initIterate, creator.Individual, lambda: make_random_individual(H, p))
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)

    # Fitness
    toolbox.register("evaluate", fitness_function_factory(D, q, C, bigM=1e9, unserved_penalty=1e3))

    # Selección, cruce, mutación
    toolbox.register("select", tools.selTournament, tournsize=3)
    toolbox.register("mate", cx_set_based, nH=H, p=p)
    toolbox.register("mutate", mut_replace_gene, nH=H, p=p, indpb=0.2)

    return toolbox

# -----------------------------------------------------------
# 6) EJECUTORES DE ALGORTIMOS DEAP (varios sabores)
# -----------------------------------------------------------



